In [2]:
import numpy as np
import polars as pl
import pandas as pd
import os
import sys
from pathlib import Path
import duckdb
main_dir = str(Path.cwd())
if 'notebooks' in str(Path.cwd()):
	parent = str(Path.cwd().parent)

sys.path.insert(1, parent)

In [3]:
RAW_DATA = "/endosome/work/InternalMedicine/s223850/Sepsis-data/data/raw_data_phase4"
RAW_DATA_PREV = "/endosome/work/InternalMedicine/s223850/Sepsis-data/data/raw_data_phase3"

In [4]:
os.listdir(RAW_DATA_PREV)

['Lab Results - 6.5.26.csv',
 'Procedure Order Events - 6.5.26.csv',
 'Med Admin Events - 6.8.26.csv',
 'flowsheets_bp - 6.8.26.csv',
 'labs_with_fio2 - 6.9.26.csv',
 'Med Admin Events - 6.5.26.csv',
 'Blood Pressure - 6.8.26.csv',
 'Export Log.xlsx',
 'Encounter Table with Baseline Values - June 2022 - May 2026 - 6.5.26.csv',
 'Diagnoses - 6.5.26.csv',
 'Flowsheet Events - 6.5.26.csv',
 'FIO2 - 6.9.26.csv']

In [5]:
df_enc_old = pl.read_csv(os.path.join(RAW_DATA_PREV, "Encounter Table with Baseline Values - June 2022 - May 2026 - 6.5.26.csv"))

In [6]:
os.listdir(RAW_DATA	)

['Lab Results - 10.8.26.csv',
 'Source of Infection Questions - 10.8.26.csv',
 'Diagnosis Events - 10.8.26.csv',
 'Flowsheet SIRS Vitals - 10.8.26.csv',
 'Encounter Table with Baseline Values - June 2022 - May 2026 - 6.5.26.csv',
 'Med Orders - 10.8.26.csv',
 'Procedure Orders - 10.8.26.csv',
 'Flowsheet Other - 10.8.26.csv',
 'Sepsis Events - 10.8.26.csv',
 'Med Admin - 10.8.26.csv']

In [4]:
HEADERS = ["EncounterEpicCsn", "Event_DateTime", "Event_ID", "Type", "Event_Grouper", "Event_Name", "NumericValue", "Value"]

In [5]:
df_labs  = pl.read_csv(os.path.join(RAW_DATA, "Lab Results - 10.8.26.csv"),
                                 has_header=False,
                                 new_columns=HEADERS,
                                 null_values=["NULL", "Null", 'null'], ignore_errors=True)
df_diag  = pl.read_csv(os.path.join(RAW_DATA, "Diagnosis Events - 10.8.26.csv"),
                                 has_header=False,
                                 new_columns=HEADERS,
                                 null_values=["NULL", "Null", 'null'], ignore_errors=True)
df_med_ord  = pl.read_csv(os.path.join(RAW_DATA, "Med Orders - 10.8.26.csv"),
                                 has_header=False,
                                 new_columns=HEADERS,
                                 null_values=["NULL", "Null", 'null'], ignore_errors=True)
df_med_admin  = pl.read_csv(os.path.join(RAW_DATA, "Med Admin - 10.8.26.csv"),
                                 has_header=False,
                                 new_columns=HEADERS,
                                 null_values=["NULL", "Null", 'null'], ignore_errors=True)
df_infect  = pl.read_csv(os.path.join(RAW_DATA, "Source of Infection Questions - 10.8.26.csv"),
                                 has_header=False,
                                 new_columns=HEADERS,
                                 null_values=["NULL", "Null", 'null'], ignore_errors=True)

## Event ID matching

## ID matching between culture and culture results

In [9]:
df_labs = df_labs.with_columns(
    pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")
)

/tmp/ipykernel_1454055/2685195108.py:2: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")


In [7]:
list(filter(lambda x: 'culture' in x.lower(), map(lambda x: x, df_labs['Event_Grouper'].unique())))

['Blood Culture Results',
 'Other Body Fluid Culture Orders',
 'Sepsis Body Fluid Culture Orders']

In [10]:
df_cultures_ord = df_labs.filter(
	pl.col("Event_Grouper").is_in(["Other Body Fluid Culture Orders", "Sepsis Body Fluid Culture Orders"])
)

df_culture_res = df_labs.filter(
	pl.col("Event_Grouper").is_in(["Blood Culture Results"])
)

In [11]:
df_cultures_ord

EncounterEpicCsn,Event_DateTime,Event_ID,Type,Event_Grouper,Event_Name,NumericValue,Value
i64,datetime[μs],i64,str,str,str,f64,str
670975883,2022-10-19 05:01:00,502769763,"""Lab Results""","""Other Body Fluid Culture Order…","""MOLDX GI ENTERIC BACTERIA PCR …",null,"""Dwight Oliver, MD"""
670975883,2022-10-19 05:01:00,502769763,"""Lab Results""","""Other Body Fluid Culture Order…","""MOLDX GI ENTERIC BACTERIA PCR …",null,"""Testing was done on this enter…"
670815148,2022-10-19 11:33:00,502818896,"""Lab Results""","""Other Body Fluid Culture Order…","""MOLDX GI ENTERIC BACTERIA PCR …",null,"""Dwight Oliver, MD"""
670815148,2022-10-19 11:33:00,502818896,"""Lab Results""","""Other Body Fluid Culture Order…","""MOLDX GI ENTERIC BACTERIA PCR …",null,"""Testing was done on this enter…"
670059200,2022-10-17 00:43:00,502415762,"""Lab Results""","""Other Body Fluid Culture Order…","""MOLDX GI ENTERIC BACTERIA PCR …",null,"""Testing was done on this enter…"
…,…,…,…,…,…,…,…
758860892,2026-08-14 11:47:00,717718941,"""Lab Results""","""Sepsis Body Fluid Culture Orde…","""GRAM ST""",null,""">10/LOW POWER FIELD Squamous …"
759252867,2026-08-14 09:28:00,717713810,"""Lab Results""","""Other Body Fluid Culture Order…","""VIRAL CULTURE, RESPIRATORY FIN…",null,"""SEE NOTE"""
759252867,2026-08-14 09:28:00,717713251,"""Lab Results""","""Sepsis Body Fluid Culture Orde…","""CULTURE""",null,"""No Nocardia species Isolated"""


In [ ]:
# df_labs_orders = df_labs.filter(
#     pl.col("Event_Grouper").str.contains(r"(?i)order")
#     &pl.col("Event_Grouper").str.contains(r"(?i)culture")
# )

# df_labs_res = df_labs.filter(
#     pl.col("Event_Grouper").str.contains(r"(?i)culture")
#     &pl.col("Event_Grouper").str.contains(r"(?i)result")
# )

In [22]:
df_labs['Type'].unique()

Type
str
"""Lab Results"""


In [23]:
df_labs.filter(
    pl.col("EncounterEpicCsn") == 714601980
).filter(
    pl.col("Event_ID").is_duplicated()
)

EncounterEpicCsn,Event_DateTime,Event_ID,Type,Event_Grouper,Event_Name,NumericValue,Value
i64,datetime[μs],i64,str,str,str,f64,str
714601980,2024-11-03 03:46:00,602752792,"""Lab Results""","""Platelets""","""PLATELETS""",140.0,"""140"""
714601980,2024-11-03 03:46:00,602752792,"""Lab Results""","""WBC""","""WBC""",6.45,"""6.45"""
714601980,2024-10-29 03:17:00,601876282,"""Lab Results""","""WBC""","""WBC""",8.43,"""8.43"""
714601980,2024-10-29 03:17:00,601876282,"""Lab Results""","""Platelets""","""PLATELETS""",160.0,"""160"""
714601980,2024-11-02 03:54:00,602696970,"""Lab Results""","""Platelets""","""PLATELETS""",154.0,"""154"""
…,…,…,…,…,…,…,…
714601980,2024-10-25 23:48:00,601513729,"""Lab Results""","""Viral Panel Components""","""CORONAVIRUS OC43""",null,"""NOT DETECTED"""
714601980,2024-10-25 23:48:00,601513729,"""Lab Results""","""Viral Panel Components""","""CORONAVIRUS 229E""",null,"""NOT DETECTED"""
714601980,2024-10-26 12:28:00,601548529,"""Lab Results""","""Sepsis Body Fluid Culture Orde…","""GLUCOSE UA""",null,"""Negative"""


In [ ]:
df_labs.filter(
    pl.col("EncounterEpicCsn").is_in([714601980])
).join(
df_labs.filter(
    pl.col("EncounterEpicCsn").is_in([714601980])
), on=['EncounterEpicCsn', 'Event_ID'],
how='inner'
).filter(
    pl.col("Event_DateTime") != pl.col("Event_DateTime_right")
)

EncounterEpicCsn,Event_DateTime,Event_ID,Type,Event_Grouper,Event_Name,NumericValue,Value,Event_DateTime_right,Type_right,Event_Grouper_right,Event_Name_right,NumericValue_right,Value_right
i64,datetime[μs],i64,str,str,str,f64,str,datetime[μs],str,str,str,f64,str


In [17]:
df_labs.group_by(
    "EncounterEpicCsn"
).agg(
    pl.len().alias('n_rows'),
    pl.col("Event_ID").n_unique().alias('n_event_id')
)

EncounterEpicCsn,n_rows,n_event_id
i64,u64,u64
671117791,118,57
735438451,422,195
730727172,233,112
748063414,41,19
760058928,226,89
…,…,…
683414264,211,90
714601980,45,14
738146418,45,20


In [15]:
sql = f"""
SELECT 
	lo.EncounterEpicCsn,
	lo.Event_ID,
    lo.Event_DateTime AS order_time,
	lo.Event_Grouper AS order_grouper,
	lo.Event_Name AS order_name,
	lo.Value AS order_value,
    lr.Event_DateTime AS res_time,
	lr.Event_Grouper AS res_grouper,
	lr.Event_Name AS res_name,
	lr.Value AS res_value
FROM lab_res as lr
JOIN lab_ord as lo
ON lr.EncounterEpicCsn=lo.EncounterEpicCsn AND lr.Event_ID=lo.Event_ID
"""

with duckdb.connect() as conn:
    conn.register("lab_res", df_culture_res)
    conn.register("lab_ord", df_cultures_ord)
    df_joined =conn.execute(sql).pl()


In [16]:
df_joined

EncounterEpicCsn,Event_ID,order_time,order_grouper,order_name,order_value,res_time,res_grouper,res_name,res_value
i64,i64,datetime[μs],str,str,str,datetime[μs],str,str,str


### Conclusion
- labs table contains only lab results. 
- `Event_ID` is probably duplicated for lab components of the same order.
- I can not join "Blood culture" Time with "Blood Culture results" time using this table

## Matching Event ID betwen Med orders and Med Admin

In [25]:
df_med_admin = df_med_admin.with_columns(
    pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")
)
df_med_ord =df_med_ord.with_columns(
    pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")
)

/tmp/ipykernel_1454055/373078599.py:2: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")
/tmp/ipykernel_1454055/373078599.py:5: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")


In [26]:
sample_encs = df_med_ord['EncounterEpicCsn'].sample(n=10, with_replacement=False, shuffle=True, seed=123534289532)
df_med_ord = df_med_ord.filter(
    pl.col("EncounterEpicCsn").is_in(sample_encs)
)
df_med_admin = df_med_admin.filter(
    pl.col("EncounterEpicCsn").is_in(sample_encs)
)

/tmp/ipykernel_1454055/2776941278.py:2: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_med_ord = df_med_ord.filter(
/tmp/ipykernel_1454055/2776941278.py:5: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_med_admin = df_med_admin.filter(


In [28]:
df_med_ord.join(
    df_med_admin,
    on=['EncounterEpicCsn', 'Event_ID'],
    how='inner',
    suffix='_admin'
).filter(
    pl.col("Event_DateTime") != pl.col("Event_DateTime_admin")
)

EncounterEpicCsn,Event_DateTime,Event_ID,Type,Event_Grouper,Event_Name,NumericValue,Value,Event_DateTime_admin,Type_admin,Event_Grouper_admin,Event_Name_admin,NumericValue_admin,Value_admin
i64,datetime[μs],i64,str,str,str,str,str,datetime[μs],str,str,str,f64,str
693965993,2023-12-20 18:20:00,557359017,"""Medication Order""","""Perioperative Antibiotics""","""PIPERACILLIN-TAZOBACTAM 3.375G…",null,null,2023-12-22 14:34:08,"""Medication Administration""","""Perioperative Antibiotics""","""PIPERACILLIN-TAZOBACTAM 3.375G…",0.0,"""g"""
693965993,2023-12-20 18:20:00,557359017,"""Medication Order""","""Perioperative Antibiotics""","""PIPERACILLIN-TAZOBACTAM 3.375G…",null,null,2023-12-23 01:25:47,"""Medication Administration""","""Perioperative Antibiotics""","""PIPERACILLIN-TAZOBACTAM 3.375G…",3.375,"""g"""
693965993,2023-12-20 18:20:00,557359017,"""Medication Order""","""Perioperative Antibiotics""","""PIPERACILLIN-TAZOBACTAM 3.375G…",null,null,2023-12-23 03:00:00,"""Medication Administration""","""Perioperative Antibiotics""","""PIPERACILLIN-TAZOBACTAM 3.375G…",null,"""*Unspecified"""
693965993,2023-12-20 18:20:00,557359017,"""Medication Order""","""Perioperative Antibiotics""","""PIPERACILLIN-TAZOBACTAM 3.375G…",null,null,2023-12-23 04:00:00,"""Medication Administration""","""Perioperative Antibiotics""","""PIPERACILLIN-TAZOBACTAM 3.375G…",null,"""*Unspecified"""
693965993,2023-12-20 18:20:00,557359017,"""Medication Order""","""Perioperative Antibiotics""","""PIPERACILLIN-TAZOBACTAM 3.375G…",null,null,2023-12-23 10:17:54,"""Medication Administration""","""Perioperative Antibiotics""","""PIPERACILLIN-TAZOBACTAM 3.375G…",3.375,"""g"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
705630549,2024-07-23 08:29:00,586232038,"""Medication Order""","""IV Antibiotics""","""MEROPENEM (MERREM) 2 G IV PUSH""",null,null,2024-08-01 01:32:00,"""Medication Administration""","""IV Antibiotics""","""MEROPENEM (MERREM) 2 G IV PUSH""",2.0,"""g"""
705630549,2024-07-23 08:29:00,586232038,"""Medication Order""","""IV Antibiotics""","""MEROPENEM (MERREM) 2 G IV PUSH""",null,null,2024-07-24 17:30:00,"""Medication Administration""","""IV Antibiotics""","""MEROPENEM (MERREM) 2 G IV PUSH""",2.0,"""g"""
705630549,2024-07-23 08:29:00,586232038,"""Medication Order""","""IV Antibiotics""","""MEROPENEM (MERREM) 2 G IV PUSH""",null,null,2024-07-26 00:30:00,"""Medication Administration""","""IV Antibiotics""","""MEROPENEM (MERREM) 2 G IV PUSH""",2.0,"""g"""


### Conclusion
Med orders and Med admins can be joined together via Event_ID

## Encounters matching
Confirm that the encounters in the new dataset exist in the old dataset

In [30]:
df_all = pl.concat([
    df_med_admin,
    df_med_ord,
    df_labs,
    df_diag,
    df_infect,
], how='vertical_relaxed')

In [32]:
all_encs = df_all['EncounterEpicCsn'].unique()
len(all_encs)

15063

In [33]:
df_all = df_all.with_columns(
    pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")
)

/tmp/ipykernel_1454055/695550849.py:2: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f")


In [35]:
df_all['Event_DateTime'].min(), df_all['Event_DateTime'].max()

(datetime.datetime(2022, 10, 1, 4, 52),
 datetime.datetime(2026, 10, 9, 3, 7, 11))

In [46]:
old_encs = df_enc_old['EncounterEpicCsn'].unique()

In [52]:
len(set(all_encs).intersection(old_encs)), len(set(all_encs).intersection(old_encs))/len(old_encs)

(13763, 0.9228241920343302)

In [53]:
len(set(all_encs).difference(set(old_encs))), len(set(all_encs).difference(set(old_encs)))/len(old_encs	)

(1300, 0.08716642081265924)

### Conclusion
We need to pull the Encounter level data for 1300 encounters

## Configurations
Confirm that the required Event_Groupers, types for the analysis exist in the new dataset.

### SIRS calculations

In [60]:
from src_strategy.data_ingest.dataloader_1 import DataLoader

In [ ]:
from src_strategy.configs.sirscalculator import sirs_config 
from src_strategy.configs

In [58]:
sirs_config

SIRSConfig(encounter_col='EncounterEpicCsn', event_dt_col='Event_DateTime', event_name_col='Event_Name', grouper_col='Event_Grouper', type_col='Type', val_col='NumericValue', raw_val_col='Value', BASELINE_CREATININE='Baseline_Creatinine', BASELINE_SBP='Baseline_SBP', BASELINE_RESP_RATE='Baseline_RespiratoryRate', BASELINE_PULSE_RATE='Baseline_PulseRate', BASELINE_PLATELETS='Baseline_Platelets', BASELINE_BILIRUBIN='Baseline_Bilirubin', BASELINE_eGFR='Baseline_eGFR', BASELINE_WBC='Baseline_WBC', temp_feature_col=<FeatureColumn.LAST_TEMP_8H: 'last_temp_8h'>, hr_feature_col=<FeatureColumn.LAST_PULSE_8H: 'last_pulse_8h'>, resp_feature_col=<FeatureColumn.LAST_RESP_8H: 'last_resp_8h'>, wbc_feature_col=<FeatureColumn.LAST_WBC_12H: 'last_wbc_12h'>, temp_flag_col='Temp_Abnormal_Flag', hr_flag_col='HR_High_Flag', resp_flag_col='Resp_Rate_High_Flag', wbc_flag_col='WBC_Abnormal_Flag', temp_lower_threshold=96.8, temp_upper_threshold=100.4, hr_upper_threshold=90.0, resp_upper_threshold=20.0, wbc_lowe

In [ ]:
DataLoader()